In [12]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch
import pandas as pd


In [16]:

# ==============================
# 1. LOAD DATA
# ==============================
df_train = pd.read_csv("../data/clean_train.csv")
df_val   = pd.read_csv("../data/clean_val.csv")


In [17]:

# ==============================
# 2. TOKENIZER + MODEL
# ==============================
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

e:\anaconda\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\saksh\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:

# ==============================
# 3. ENCODING
# ==============================
def encode_texts(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=64   
    )

train_encodings = encode_texts(df_train['tweet_content'].tolist())
val_encodings   = encode_texts(df_val['tweet_content'].tolist())

train_labels = torch.tensor(df_train['label'].values)
val_labels   = torch.tensor(df_val['label'].values)


In [20]:

# ==============================
# 4. DATASET CLASS
# ==============================
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset   = SentimentDataset(val_encodings, val_labels)


In [21]:

# ==============================
# 5. TRAINING ARGUMENTS
# ==============================
training_args = TrainingArguments(
    output_dir='../models/distilbert_model',
    num_train_epochs=2,                 
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",                 
    logging_steps=100,
    fp16=True                           
)


In [22]:

# ==============================
# 6. TRAINER
# ==============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:

# ==============================
# 7. TRAIN + EVALUATE
# ==============================
trainer.train()
trainer.evaluate()


In [ ]:

# ==============================
# 8. SAVE MODEL
# ==============================
model.save_pretrained("../models/distilbert_model")
tokenizer.save_pretrained("../models/distilbert_model")